# 01 · Input & Inspect  —  EDA Step 2

**Goal:** load *one* month of RTT data and confirm three things before we build the full loader:
1. `Part_3` (New RTT Periods) exists — our **demand** signal.
2. How to tell **NHS trusts** (`R##` codes) from **independent** hospitals.
3. The `C_999` (Total) rows we must **drop** so we don't double-count.

Kernel should be **FedOPD (.venv)** (top-right). Run each cell with **Shift+Enter**.
Log what you find in `docs/EDA_LOG.md` → Step 2.

## Cell 1 — setup: imports + list the 60 files

- `from pathlib import Path` — `Path` is a modern, tidy way to handle file paths (nicer than string concatenation).
- `import pandas as pd` — pandas is the table/dataframe library; `pd` is the conventional short alias.
- `DATA_DIR = Path(r"...")` — the folder holding the 60 CSVs. The `r"..."` is a **raw string** so Windows backslashes aren't treated as escape characters.
- `DATA_DIR.glob("rtt_*.csv")` — finds every file matching that pattern (`*` = wildcard).
- `sorted(...)` — puts them in filename order, which for `rtt_YYYY-MM.csv` is also **date** order.
- `files[0]` / `files[-1]` — first and last items (`-1` = last). `.name` is just the filename without the folder.

In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path(r"C:\Users\heave\Documents\FedOPD\data\nhs_rtt_referral_to_treatment")
files = sorted(DATA_DIR.glob("rtt_*.csv"))

print(len(files), "files")
print("first:", files[0].name)
print("last :", files[-1].name)

60 files
first: rtt_2021-04.csv
last : rtt_2026-03.csv


## Cell 2 — load ONE month and look at its shape

- `pd.read_csv(path)` — reads a CSV file into a **DataFrame** (an in-memory table).
- `files[-1]` — the most recent month (`2026-03`). We start with one file, not all 60, so it's fast.
- `low_memory=False` — tells pandas to read the whole file before deciding each column's type (avoids mixed-type warnings on big files).
- `df.shape` — a `(rows, columns)` tuple. Expect ~183,000 rows × 121 columns.
- `df.head()` — shows the first 5 rows so you can see the real data.

In [2]:
df = pd.read_csv(files[-1], low_memory=False)   # files[-1] = latest month (2026-03)

print("shape (rows, columns):", df.shape)
df.head()

shape (rows, columns): (183400, 121)


,Period,Provider Parent Org Code,Provider Parent Name,Provider Org Code,Provider Org Name,Commissioner Parent Org Code,Commissioner Parent Name,Commissioner Org Code,Commissioner Org Name,RTT Part Type,...,Gt 98 To 99 Weeks SUM 1,Gt 99 To 100 Weeks SUM 1,Gt 100 To 101 Weeks SUM 1,Gt 101 To 102 Weeks SUM 1,Gt 102 To 103 Weeks SUM 1,Gt 103 To 104 Weeks SUM 1,Gt 104 Weeks SUM 1,Total,Patients with unknown clock start date,Total All
0,RTT-March-2026,QE1,NHS LANCASHIRE AND SOUTH CUMBRIA INTEGRATED CA...,A4M8P,BUCKSHAW HOSPITAL,NaN,NaN,13Q,NATIONAL COMMISSIONING HUB 1,Part_2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,1
1,RTT-March-2026,QE1,NHS LANCASHIRE AND SOUTH CUMBRIA INTEGRATED CA...,A4M8P,BUCKSHAW HOSPITAL,NaN,NaN,13Q,NATIONAL COMMISSIONING HUB 1,Part_2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,1
2,RTT-March-2026,QE1,NHS LANCASHIRE AND SOUTH CUMBRIA INTEGRATED CA...,A4M8P,BUCKSHAW HOSPITAL,NaN,NaN,13Q,NATIONAL COMMISSIONING HUB 1,Part_2,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,NaN,NaN,2
3,RTT-March-2026,QE1,NHS LANCASHIRE AND SOUTH CUMBRIA INTEGRATED CA...,A4M8P,BUCKSHAW HOSPITAL,NaN,NaN,13Q,NATIONAL COMMISSIONING HUB 1,Part_3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
4,RTT-March-2026,QE1,NHS LANCASHIRE AND SOUTH CUMBRIA INTEGRATED CA...,A4M8P,BUCKSHAW HOSPITAL,NaN,NaN,13Q,NATIONAL COMMISSIONING HUB 1,Part_3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1


## Cell 3 — pathway types (confirm `Part_3` = demand)

Each row reports one **pathway measure** for a provider×specialty. RTT has several. We want the *new demand* one.

- `df[["col1", "col2"]]` — the **double** brackets select multiple columns as a smaller table.
- `.drop_duplicates()` — collapses to the **distinct** rows, so we see each pathway type once.

**Look for** `Part_3  →  New RTT Periods - All Patients` — that's the demand signal we'll forecast. (`Part_2` = the current waiting list; `Part_1A/1B` = completed pathways.)

In [3]:
df[["RTT Part Type", "RTT Part Description"]].drop_duplicates()

,RTT Part Type,RTT Part Description
0,Part_2,Incomplete Pathways
3,Part_3,New RTT Periods - All Patients
12,Part_1A,Completed Pathways For Admitted Patients
15,Part_1B,Completed Pathways For Non-Admitted Patients
24,Part_2A,Incomplete Pathways with DTA


## Cell 4 — separate NHS trusts from independents by code shape

Provider **org codes** encode the organisation type. NHS trusts use short **3-character** codes starting with `R`; independent (private) hospitals use longer **5-character** codes. Let's verify that.

- `df["Provider Org Code"]` — selects one column (a pandas **Series**).
- `.dropna()` — removes blank/missing values.
- `.unique()` — keeps each distinct code once (we care about the *set* of providers, not every row).
- `[len(c) for c in codes]` — a **list comprehension**: the length of each code.
- `pd.Series(...).value_counts()` — tallies how many codes have each length (3, 5, …).
- `is_trust = lambda c: ...` — a tiny inline function returning **True** if the code looks like an NHS trust (3 chars, starts `R`).
- `sum(is_trust(c) for c in codes)` — counts how many pass the test (`True` counts as 1).
- The last two prints show a few example codes of each kind so you can eyeball them.

In [4]:
codes = df["Provider Org Code"].dropna().unique()

lengths = pd.Series([len(c) for c in codes]).value_counts()
print("provider-code length -> count:")
print(lengths, "\n")

is_trust = lambda c: len(c) == 3 and c[0] == "R"   # NHS trust / foundation-trust pattern
n_trust = sum(is_trust(c) for c in codes)
print(f"{n_trust} of {len(codes)} providers look like NHS trusts (R##)\n")

print("sample trusts      :", [c for c in codes if is_trust(c)][:5])
print("sample independents:", [c for c in codes if not is_trust(c)][:5])

provider-code length -> count:
5    339
3    194
Name: count, dtype: int64 

149 of 533 providers look like NHS trusts (R##)

sample trusts      : ['RTX', 'RW5', 'RXL', 'RXN', 'RXR']
sample independents: ['A4M8P', 'A5E1F', 'AAV', 'C3Y0A', 'NP9']


## Cell 5 — treatment functions, incl. the `Total` (C_999) we drop

`Treatment Function` = the clinical **specialty** (Cardiology, Urology, …) — this is our `dept`.

- Same `[[...]].drop_duplicates()` trick to list the distinct specialties.
- **Watch for** `C_999  →  Total`: that's an **aggregate** row (the sum of all specialties). If we keep it we'd double-count demand, so the loader will filter it out.

In [5]:
df[["Treatment Function Code", "Treatment Function Name"]].drop_duplicates()

,Treatment Function Code,Treatment Function Name
0,C_100,General Surgery Service
1,C_101,Urology Service
2,C_999,Total
10,C_110,Trauma and Orthopaedic Service
13,C_502,Gynaecology Service
20,C_120,Ear Nose and Throat Service
21,C_301,Gastroenterology Service
72,X06,Other - Other Services
210,X02,Other - Medical Services
297,C_130,Ophthalmology Service


### Findings (fill in as you run)
- Part types seen — is `Part_3` present?: 
- Provider-code length split / #trusts vs independents: 
- `C_999 = Total` present?: 

→ **Next:** Cell 6+ = the canonical loader — trusts-only `[date, branch, dept, demand]` across all 60 months.